# OXIOW — Teste 1: SoulX-FlashHead na GPU grátis (Colab T4)

**O que este notebook faz:** gera um vídeo de avatar falante usando **NOSSO rosto** e
**NOSSA fala (Margaret)**, no critério OXIOW de qualidade (sem visual de IA).

**Antes de rodar (obrigatório):**
1. Menu **Runtime → Change runtime type**
2. Em *Hardware accelerator* escolha **T4 GPU**
3. Clique **Save**
4. Depois: menu **Runtime → Run all** (ou Ctrl+F9)

**Tempo esperado:** ~20 min (14,7 GB de pesos na primeira vez + inferência).
**Custo:** R$ 0 — GPU gratuita do Colab.

**O que sai no final:** `flashhead-9x16-limpo.mp4` — vertical, sem metadados,
pronto para subir em tráfego pago.

---

### Régua de avaliação (o que olhar no vídeo)

| # | Critério | O que queremos |
|---|---|---|
| 1 | Boca | **sutil**, não teatral |
| 2 | Pisca | frequência humana, não robótica |
| 3 | Respiração | visível, na fala E no silêncio |
| 4 | Pele | naturalista, sem "alisamento de IA" |
| 5 | Pose | pessoa normal, sem gesticulação exagerada |
| 6 | Identidade | estável, não "derrete" |


In [ ]:
# ══ 1 · AMBIENTE — confere ANTES de gastar 20 min instalando ══
import os, sys, subprocess, glob, time, shutil, re, textwrap

RAIZ = "/content" if os.path.isdir("/content") else (
       "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd())
os.chdir(RAIZ)
REPO = os.path.join(RAIZ, "flashhead")

def run(cmd, cwd=None, timeout=5400, mostrar=True, tolerante=False):
    r = subprocess.run(cmd, shell=isinstance(cmd, str), cwd=cwd,
                       capture_output=True, text=True, timeout=timeout)
    out = (r.stdout or "") + (r.stderr or "")
    if mostrar:
        print(out[-3000:], flush=True)
    if r.returncode != 0 and not tolerante:
        print(f"\n   >>> FALHOU (exit {r.returncode})")
        raise SystemExit(1)
    return r.returncode, out

gpu, vram = "nenhuma", 0
try:
    _, o = run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", mostrar=False)
    L = [x for x in o.strip().splitlines() if x.strip()]
    if L:
        gpu = L[0]
        m = re.search(r"(\d+)\s*MiB", gpu)
        if m: vram = int(m.group(1))
except Exception as e:
    print("   nvidia-smi:", e)

import torch
print(f"   GPU .........: {gpu}")
print(f"   VRAM ........: {vram/1024:.1f} GB")
print(f"   torch .......: {torch.__version__}   <-- NAO MEXER")
try:
    import torchaudio; import torchaudio.functional
    print(f"   torchaudio ..: {torchaudio.__version__} ok")
except Exception as e:
    print(f"   torchaudio ..: QUEBRADO -> consertando ({str(e)[:70]})")
    run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall",
         f"torchaudio=={torch.__version__.split('+')[0]}"], tolerante=True)

if not torch.cuda.is_available():
    print("\n   >>> SEM GPU. Faca: Runtime -> Change runtime type -> T4 GPU -> Run all")
    raise SystemExit(1)
print(f"   disco livre .: {shutil.disk_usage(RAIZ).free/1e9:.1f} GB")
print("=== CHECKPOINT 1 OK ===")


In [ ]:
# ══ 2 · REPO OFICIAL + DEPENDENCIAS (SEM TOCAR NO TORCH) ══
if not os.path.isdir(REPO):
    run(f"git clone --depth 1 https://github.com/Soul-AILab/SoulX-FlashHead.git {REPO}")
os.chdir(REPO)
for f in ("generate_video.py", "requirements.txt"):
    print(f"   {f:24s}: {'OK' if os.path.exists(f) else 'AUSENTE'}")

# trava: impede o pip de trocar torch/torchaudio (a causa do 'undefined symbol' na v1)
trava = os.path.join(RAIZ, "trava.txt")
with open(trava, "w") as f:
    f.write(f"torch=={torch.__version__}\n")
    try: f.write(f"torchaudio=={torchaudio.__version__}\n")
    except Exception: f.write(f"torchaudio=={torch.__version__}\n")
print(f"   trava: torch=={torch.__version__}")

rc, _ = run([sys.executable, "-m", "pip", "install", "-q", "-c", trava,
             "-r", "requirements.txt"], tolerante=True, mostrar=False)
print("   requirements:", "OK" if rc == 0 else "algum falhou (seguindo)")

for p in ["opencv-python>=4.12.0.88", "diffusers>=0.34.0", "transformers==4.57.3",
          "tokenizers>=0.20.3", "accelerate>=1.8.1", "tqdm", "imageio", "easydict",
          "ftfy", "imageio-ffmpeg", "scikit-image", "loguru", "pyloudnorm",
          "decord", "librosa", "flask", "edge-tts", "soundfile"]:
    rc, _ = run([sys.executable, "-m", "pip", "install", "-q", "-c", trava, p],
                tolerante=True, mostrar=False)
    print(f"   {'OK    ' if rc == 0 else 'FALHOU'} {p}")

# xformers: opcional
rc, _ = run([sys.executable, "-m", "pip", "install", "-q", "-c", trava, "xformers==0.0.31"],
            tolerante=True, mostrar=False)
print(f"   {'OK    ' if rc==0 else 'NAO (ok, usa SDPA)'} xformers")

# mediapipe sem wheel -> shim (nao usamos face_crop)
try:
    import mediapipe  # noqa
    print("   mediapipe ...: OK")
except Exception:
    shim = os.path.join(RAIZ, "_shim"); os.makedirs(shim, exist_ok=True)
    with open(os.path.join(shim, "mediapipe.py"), "w") as f:
        f.write('# Shim: o repo importa mediapipe so para face_crop, que nao usamos.\n'
                '__version__ = "0.0.0-shim"\n'
                'class _Indisponivel:\n'
                '    def __init__(self, *a, **k):\n'
                '        raise RuntimeError("mediapipe indisponivel; rode SEM --use_face_crop")\n'
                'class solutions:\n'
                '    face_detection = _Indisponivel\n')
    if shim not in sys.path: sys.path.insert(0, shim)
    os.environ["PYTHONPATH"] = shim + os.pathsep + os.environ.get("PYTHONPATH", "")
    print(f"   mediapipe ...: SHIM em {shim}")

print("\n   --- conferindo os imports que importam ---")
for mod in ["torch", "torchaudio", "cv2", "diffusers", "transformers",
            "librosa", "decord", "easydict", "pyloudnorm"]:
    try:
        __import__(mod); print(f"      {mod:14s} OK")
    except Exception as e:
        print(f"      {mod:14s} falhou ({str(e)[:55]})")
print("=== CHECKPOINT 2 OK ===")


In [ ]:
# ══ 3 · NOSSO ROSTO E NOSSA FALA (a identidade OXIOW) ══
TEMP = os.path.join(RAIZ, "_temp"); os.makedirs(TEMP, exist_ok=True)

# --- o rosto (o MESMO usado no Teste 4, para o comparativo ser justo) ---
ROSTO = os.path.join(TEMP, "rosto.png")
URL_ROSTO = ("https://raw.githubusercontent.com/guilefranca-cyber/"
             "oxiow-gerador-notebooks/main/assets/avatar-dona-maria.png")
if not (os.path.exists(ROSTO) and os.path.getsize(ROSTO) > 100_000):
    run(f"curl -sL -o {ROSTO} {URL_ROSTO}", tolerante=True, mostrar=False)
print(f"   rosto: {ROSTO} ({os.path.getsize(ROSTO)/1024:.0f} KB)"
      if os.path.exists(ROSTO) else "   🔴 rosto NAO baixou")
if not os.path.exists(ROSTO):
    raise SystemExit("sem rosto — nao da para gerar")

# --- a fala (mesma do Teste 4) ---
FALA = ("Hi, I'm Margaret. I'm sixty-two years old. For years my feet ached "
        "every single evening. I tried creams, I tried soaking them, and nothing "
        "really helped. Then a friend told me about something simple. "
        "If your feet bother you too, stay with me for a moment.")
AUDIO = os.path.join(TEMP, "fala-margaret.mp3")
if not (os.path.exists(AUDIO) and os.path.getsize(AUDIO) > 5000):
    code = ("import asyncio, edge_tts\n"
            "async def m():\n"
            f"    c = edge_tts.Communicate({FALA!r}, 'en-US-AriaNeural')\n"
            f"    await c.save({AUDIO!r})\n"
            "asyncio.run(m())\n")
    run([sys.executable, "-c", code], tolerante=True, mostrar=False)
print(f"   fala : {AUDIO} ({os.path.getsize(AUDIO)/1024:.0f} KB)"
      if os.path.exists(AUDIO) else "   🔴 fala NAO gerou")
if not os.path.exists(AUDIO):
    raise SystemExit("sem fala — nao da para gerar")

_, o = run(f"ffprobe -v error -show_entries format=duration -of csv=p=0 {AUDIO}",
           tolerante=True, mostrar=False)
print(f"   duracao da fala: {o.strip()[:12]} s")
print("=== CHECKPOINT 3 OK ===")


In [ ]:
# ══ 4 · OS PESOS (~14,7 GB — o download principal) ══
MODELOS = os.path.join(REPO, "models"); os.makedirs(MODELOS, exist_ok=True)
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "0")

for pasta, repo_hf in {"SoulX-FlashHead-1_3B": "Soul-AILab/SoulX-FlashHead-1_3B",
                       "wav2vec2-base-960h": "facebook/wav2vec2-base-960h"}.items():
    destino = os.path.join(MODELOS, pasta)
    ja = (glob.glob(os.path.join(destino, "**", "*.safetensors"), recursive=True) +
          glob.glob(os.path.join(destino, "**", "*.pth"), recursive=True))
    if len(ja) >= 3:
        print(f"   {pasta}: JA ESTA AQUI ({len(ja)} arquivos) — pulando")
        continue
    print(f"   {pasta}: baixando de {repo_hf} ...")
    code = ("from huggingface_hub import snapshot_download\n"
            f"snapshot_download(repo_id={repo_hf!r}, local_dir={destino!r})\n"
            "print('fim')\n")
    run([sys.executable, "-c", code], tolerante=True)

print("\n   --- pesos no lugar ---")
for f in sorted(glob.glob(os.path.join(MODELOS, "**", "*"), recursive=True)):
    if os.path.isfile(f) and os.path.getsize(f) > 50_000_000:
        print(f"      {os.path.getsize(f)/1e9:6.2f} GB  {os.path.relpath(f, MODELOS)}")
print("=== CHECKPOINT 4 OK ===")


In [ ]:
# ══ 5 · INFERENCIA — Model_Lite (o que cabe na T4) ══
CK  = os.path.join(MODELOS, "SoulX-FlashHead-1_3B")
W2V = os.path.join(MODELOS, "wav2vec2-base-960h")
SAIDA = os.path.join(RAIZ, "saida_avatar"); os.makedirs(SAIDA, exist_ok=True)

cmd = [sys.executable, "generate_video.py",
       "--ckpt_dir", CK, "--wav2vec_dir", W2V,
       "--model_type", "lite",
       "--cond_image", ROSTO,
       "--audio_path", AUDIO,
       "--audio_encode_mode", "stream"]
print("   rodando:", " ".join(cmd))
print("   (a saida aparece AO VIVO abaixo; ~10-20 min)\n")

env = dict(os.environ)
env["PYTHONPATH"] = os.pathsep.join([os.path.join(RAIZ, "_shim"), REPO,
                                     env.get("PYTHONPATH", "")])
env["PYTHONUNBUFFERED"] = "1"
t0 = time.time()
p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE,
                     stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
for linha in p.stdout:
    print("      " + linha.rstrip(), flush=True)
p.wait()
print(f"\n   exit: {p.returncode}   tempo: {(time.time()-t0)/60:.1f} min")

vids = sorted([v for v in (glob.glob(os.path.join(REPO, "**", "*.mp4"), recursive=True) +
                           glob.glob(os.path.join(RAIZ, "**", "*.mp4"), recursive=True))
               if os.path.getmtime(v) >= t0],
              key=os.path.getmtime, reverse=True)
print(f"   videos NOVOS: {len(vids)}")
if not vids:
    print("   🔴 nao gerou video — copie as 40 ultimas linhas e me mande")
    raise SystemExit(1)
BRUTO = vids[0]
print(f"   ✅ {BRUTO} ({os.path.getsize(BRUTO)/1e6:.2f} MB)")
print("=== CHECKPOINT 5 OK ===")


In [ ]:
# ══ 6 · 9:16 + METADADO LIMPO (regra permanente OXIOW) ══
VERT = os.path.join(SAIDA, "flashhead-9x16.mp4")
filtro = ("[0:v]scale=1080:1920:force_original_aspect_ratio=increase,"
          "crop=1080:1920,boxblur=40:5[bg];"
          "[0:v]scale=1080:-2[fg];"
          "[bg][fg]overlay=(W-w)/2:(H-h)/2")
run(["ffmpeg", "-y", "-loglevel", "error", "-i", BRUTO, "-filter_complex", filtro,
     "-map_metadata", "-1", "-c:v", "libx264", "-crf", "20", "-preset", "medium",
     "-pix_fmt", "yuv420p", "-c:a", "aac", "-b:a", "192k", VERT], tolerante=True)

LIMPO = os.path.join(SAIDA, "flashhead-9x16-limpo.mp4")
rc, _ = run(["ffmpeg", "-y", "-loglevel", "error", "-i", VERT, "-map_metadata", "-1",
             "-map_chapters", "-1", "-fflags", "+bitexact", "-flags:v", "+bitexact",
             "-flags:a", "+bitexact", "-c", "copy", LIMPO], tolerante=True)
if rc != 0 or not os.path.exists(LIMPO):
    shutil.copy2(VERT, LIMPO)

_, o = run(f"ffprobe -v error -show_entries format_tags -of default=nw=1 {LIMPO}",
           tolerante=True, mostrar=False)
tags = [l for l in o.splitlines() if l.strip() and "=" in l]
print(f"   metadado: {'✅ LIMPO' if not tags else tags[:4]}")
print(f"   ✅ PRONTO: {LIMPO}  ({os.path.getsize(LIMPO)/1e6:.2f} MB)")
print()
print("   >>> ABRA O VIDEO na pasta 'saida_avatar' (coluna da esquerda) <<<")
print("   e me diga olhando os 6 criterios: boca mais sutil? pisca melhor?")
print("   respiracao? pele? pose? identidade estavel?")
